<a href="https://colab.research.google.com/github/EvenSol/NeqSim-Colab/blob/master/notebooks/process/co2_chain_from_compression_to_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CO2 Chain from Compression to Pipeline

A compact CO2-rich stream example covering compression, cooling, dense-phase screening, and pipeline arrival pressure.


## Setup

Import NeqSim and build a CO2-rich fluid.


In [ ]:
# Install NeqSim when running in a fresh Colab session.
try:
    import neqsim
except ImportError:
    %pip install neqsim

import json
import pandas as pd
import matplotlib.pyplot as plt
try:
    from neqsim import jneqsim
except ImportError:
    from neqsim import jNeqSim as jneqsim
from neqsim.thermo import TPflash

SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
Stream = jneqsim.process.equipment.stream.Stream
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

## Create the CO2-Rich Fluid

The example stream contains CO2 with methane and nitrogen impurities.


In [ ]:
fluid = SystemSrkEos(273.15 + 25.0, 25.0)
fluid.addComponent("CO2", 0.96)
fluid.addComponent("methane", 0.03)
fluid.addComponent("nitrogen", 0.01)
fluid.setMixingRule("classic")
TPflash(fluid)
fluid.initProperties()


## Build Compression and Cooling

Compress the stream into a pipeline inlet condition.


In [ ]:
Compressor = jneqsim.process.equipment.compressor.Compressor
Heater = jneqsim.process.equipment.heatexchanger.Heater
feed = Stream("CO2 feed", fluid)
feed.setFlowRate(100000.0, "kg/hr")
compressor = Compressor("CO2 compressor", feed)
compressor.setOutletPressure(110.0)
cooler = Heater("Aftercooler", compressor.getOutStream())
cooler.setOutTemperature(273.15 + 25.0)
process = ProcessSystem()
for unit in [feed, compressor, cooler]:
    process.add(unit)
process.run()
out = cooler.getOutStream().getFluid()
out.initProperties()
{"pressure_bara": out.getPressure("bara"), "temperature_C": out.getTemperature("C"), "density_kg_per_m3": out.getDensity("kg/m3"), "compressor_power_kW": compressor.getPower("kW")}


## Pipeline Screening

Use a transparent pressure-gradient placeholder for early screening; detailed work should use the NeqSim pipe models.


In [ ]:
length_km = 80.0
pressure_gradient_bar_per_km = 0.08
arrival_pressure_bara = out.getPressure("bara") - length_km * pressure_gradient_bar_per_km
dense_phase_flag = out.getDensity("kg/m3") > 400.0
pd.DataFrame([{"length_km": length_km, "arrival_pressure_bara": arrival_pressure_bara, "dense_phase_screen": dense_phase_flag}])
